# Módulo 2 — Comparación de sesgo + SD1.5 antes/después

Recalcula sobre imágenes ya generadas: no vuelve a correr SAM/FRAN/FADING. APIs legacy y ablaciones permanecen separadas. El panel principal es la intersección válida, aging, balanceado por horizonte e identidad; también se reporta el panel completo por modelo.


In [ ]:
%pip install -e ".[aging-bias,evaluation]"


In [ ]:
from pathlib import Path
from dataclasses import replace
import json
import pandas as pd
from src.aging_bias import FeatureConfig, PaperV2Config, WeightingConfig, SD15Config
from src.aging_bias.references import ReferenceBundle
from src.comparisong.evaluation_backends_v2 import prepare_paper_bundle
from src.comparisong.evaluate_paper_v2 import evaluate_paper_v2

SAVED = Path("/content/drive/MyDrive/respaldo_test_comparison")
REFERENCE_DIR = Path("/content/drive/MyDrive/aging_bias_references")
OUTPUT = Path("/content/drive/MyDrive/paper_comparison_v2")
ASSETS = Path("/content/evaluation_models_v2")
MODELS = ["SAM", "FRAN", "FADING", "HRFAE", "ours_strength_0.31", "ours_picking_no_prompt", "ours_picking_prompt"]
references = ReferenceBundle.load(REFERENCE_DIR)
features = FeatureConfig(**references.metadata["config"]["features"])


## 1. Preparar evaluadores una sola vez, CPU

FairFace sustituye a DEX/MiVOLO como estimador principal de edad. DEX se conserva solo en apéndice con procedencia. FairFace no es libre de sesgo: el error relativo al target reduce efectos comunes, no garantiza cancelarlos. Su grupo abierto 70+ se representa por 85 años (convención documentada).

Puedes pasar `fairface_checkpoint_path` y `fairface_url` si ya tienes una copia verificada. AdaFace y KID se reutilizan desde las rutas anteriores; no instala py-feat/MiVOLO/timm. Si tienes OpenCV 5 o un módulo cv2 roto, resuelve la instalación 4.x y reinicia antes de ejecutar.


In [ ]:
bundle = prepare_paper_bundle(
    ASSETS, feature_config=features, allow_download=True,
    include_dex=True, include_kid=True,
    # fairface_checkpoint_path="/content/drive/MyDrive/checkpoints/fairface_resnet34.pt",
    legacy_metrics_config={
        "adaface_repo_path": "/content/evaluation_models/AdaFace",
        "adaface_checkpoint_path": "/content/evaluation_models/adaface_ir101_webface12m.ckpt",
        "dex_prototxt_path": "/content/evaluation_models/DEX/age.prototxt",
        "dex_checkpoint_path": "/content/evaluation_models/DEX/dex_chalearn_iccv2015.caffemodel",
        "kid_inception_weights_path": "/content/evaluation_models/KID/weights-inception-2015-12-05-6726825d.pth",
    },
)


## Variantes nuestras: fijo + picking sin/con prompts

Esta sección **sí genera las dos variantes nuevas**; no vuelve a ejecutar literatura ni el modelo fijo guardado. Reutiliza el bundle de inferencia de tu smart sweep, con ArcFace/MiVOLO. Es distinto de `bundle`, que contiene los evaluadores independientes FairFace/AdaFace en CPU. No utiliza la métrica de sesgo para seleccionar candidatos.

El smart sweep busca hasta 10 candidatos base entre 0.04 y 0.45 y genera una imagen asistida adicional con el strength seleccionado. Se guardan ambas, aunque la asistida sea peor. `resume=True` reutiliza resultados existentes.


In [ ]:
from src.comparisong.picking_variants import PickingVariantsConfig, run_picking_variants, default_prompt_assistance

CHECKPOINT_PATH = "/content/drive/MyDrive/adapter_inference.pt"
target_map = {8:0.33, 15:0.31, 26:0.04, 35:0.22, 44:0.24, 55:0.27, 65:0.29}
prompt_assistance_config = default_prompt_assistance()  # Editable: positivos, negativos y reglas.

# Si ya tienes el bundle usado en diagnose_checkpoint_smart_age_sweep:
# picking_inference_bundle = TU_BUNDLE_DE_INFERENCIA
# En una sesión nueva, instala primero las dependencias auxiliares del proyecto
# (las mismas de tu smart sweep; NO usar DEX/FairFace para esta calibración).
if "picking_inference_bundle" not in globals():
    from src.inference import load_face_aging_inference_bundle
    picking_inference_bundle = load_face_aging_inference_bundle(
        CHECKPOINT_PATH, device="cuda", local_files_only=False,
        load_auxiliary_models=True, auxiliary_trust_remote_code=True,
    )

picking_config = PickingVariantsConfig(
    checkpoint_path=CHECKPOINT_PATH, test_manifest_path=SAVED / "transitions.csv",
    source_root="/content/drive/MyDrive/dataset_def/test",
    output_dir=OUTPUT / "ours_picking_variants",
    target_age_strength_map=target_map, prompt_assistance_config=prompt_assistance_config,
    max_trials_per_target=10, min_strength=0.04, max_strength=0.45,
    use_bias_corrected_mivolo_target=True,
    confidence_aging=3.0, confidence_rejuvenecer=2.0,
    strength_step_coarse=0.05, strength_step_medium=0.03, strength_step_fine=0.02,
    prompt_assistance_scale=0.30, negative_prompt_assistance_scale=0.25,
    enable_guidance_micro_search=False, identity_tiebreak=True,
    identity_margin_for_tiebreak=1.0, save_all_trials=False,
    seed=2026, num_inference_steps=40, image_size=400, resume=True,
)
picking_outputs = run_picking_variants(picking_config, bundle=picking_inference_bundle)
print(picking_outputs["generated_roots"])


## 2. Panel existente + pesos operativos

No muestreamos otro test para favorecer resultados. Los bins `[1,10),[10,20),[20,30),[30,inf)` reciben igual masa y cada identidad pesa igual dentro del bin. Bins pequeños se fusionan y se documentan. `unweighted` sigue siendo la media simple. Una distribución operativa externa puede indicarse con `operational_target=[...]`.


In [ ]:
config = PaperV2Config(
    test_manifest_path=SAVED / "prediction_manifest.csv",
    generated_roots={**{m: SAVED / "generated" / m for m in MODELS}, **picking_outputs["generated_roots"]},
    models=MODELS, reference_bundle_path=REFERENCE_DIR, output_dir=OUTPUT,
    features=features,
    # Si cambiaron rutas de los originales:
    source_root="/content/drive/MyDrive/dataset_def/test",
    weighting=WeightingConfig(scheme_primary="bin_balanced", bins=(1,10,20,30), identity_balance=True, min_bin_n=10),
    n_boot=1000, seed=2026, resume=True, batch_size=16,
)
result = evaluate_paper_v2(config, bundle=bundle)
for subset in ["all", "aging"]:
    print("Envejecimiento + rejuvenecimiento" if subset == "all" else "SOLO envejecimiento")
    display(pd.read_csv(OUTPUT / f"tables/table1_bias_{subset}.csv"))
    display(pd.read_csv(OUTPUT / f"tables/table2_operational_{subset}.csv"))
    display(pd.read_csv(OUTPUT / f"tables/tableA1_legacy_{subset}.csv"))


## 3. PARADA OBLIGATORIA: revisar anclas reales

No interpretes pi/drift/homog cuando el status sea `not_identified` o `no_edit`. No recortes razones a [0,1]. Si el oráculo falla, revisa atributos, ICC, soporte y armonización; no es un fallo de los competidores. Las CI por identidad son condicionales a las referencias estimadas.


In [ ]:
anchors = json.loads((OUTPUT / "validation/real_anchors.json").read_text())
display(pd.DataFrame(anchors["checks"]))
print("Anclas válidas:", anchors["passed"])
print(anchors["note"])


## 4. Experimento central: SD1.5 antes vs después

**Ejecuta solo después de revisar las anclas.** Esta sección sí genera nuevas imágenes y puede ser costosa. No se ejecuta al importar módulos ni al recalcular métricas. Cambia `ANCHORS_REVIEWED` solo tras revisar la sección anterior.

Carga V5 una vez, guarda la conv_in preentrenada antes de aplicar el checkpoint y mantiene el modelo residente. Vanilla apaga DoRA/LoRA, fuente concatenada y edad; restaura también los cuatro canales/bias originales preentrenados. Ambas condiciones usan el mismo protocolo directo, prompts, CFG y semillas. No usa picking ni selecciona usando la métrica de sesgo.


In [ ]:
from src.comparisong.sd15_counterfactual import run_sd15_before_after
ANCHORS_REVIEWED = False
sd_config = SD15Config(
    checkpoint_path="/content/drive/MyDrive/adapter_inference.pt",
    test_manifest_path=SAVED / "transitions.csv",
    source_root="/content/drive/MyDrive/dataset_def/test",
    output_dir=OUTPUT / "sd15_counterfactual",
    validation_path=OUTPUT / "validation/real_anchors.json",
    anchors_reviewed=ANCHORS_REVIEWED,
    strengths=(0.10,0.20,0.30,0.40,0.50), seed=2026, resume=True,
    inference_config={"image_size":400,"num_inference_steps":34,
        "text_reference_mode":"source_age","age_guidance_scale":3.0,
        "text_guidance_scale":7.0,"image_guidance_scale":1.5,
        "negative_prompt":""},
    local_files_only=False,
)
sd_outputs = run_sd15_before_after(sd_config)


## 5. Evaluar SD1.5 con los mismos pares y comparar con literatura

El contrafactual genera solo aging. Se usa explícitamente ese mismo subconjunto para todos los modelos; se conservan por separado los resultados del panel original completo. Las curvas por strength no deben seleccionarse por pi.


In [ ]:
aging_manifest = OUTPUT / "sd15_counterfactual/transitions.csv"
with_sd = replace(config, test_manifest_path=aging_manifest,
    models=MODELS + list(sd_outputs["generated_roots"]),
    generated_roots={**config.generated_roots, **sd_outputs["generated_roots"]},
    output_dir=OUTPUT / "with_sd15")
sd_result = evaluate_paper_v2(with_sd, bundle=bundle)
from src.aging_bias.reporting import plot_sd15_before_after
matched_d = plot_sd15_before_after(OUTPUT / "with_sd15")
display(matched_d)


## Archivos y reanudación

Conserva toda la carpeta: `per_sample.parquet`, `bias_indices.csv`, `bias_indices_by_bin.csv`, `paired_differences.csv`, `operational_metrics.csv`, `kid_metrics.csv`, `validation/`, `tables/`, `cache/`, `run_metadata.json`. `resume=True` reutiliza features y predicciones de evaluadores por SHA-256; el bootstrap se recalcula de forma reproducible. SD1.5 reutiliza PNG+JSON válidos. No cambies firmas/manual ni mezcles protocolos.
